# Pass@1: T5 007 vs Copy 008

Both saved version2 final models. Same complete fixed validation set: 20,000 rows,
beam4, max_new_tokens=256, deterministic decoding, KV cache, FP16 on T4.
Each model uses one GPU; no training. Reference code is decoded from unchanged saved
labels. Execution uses our existing bounded evaluator, timeout2s, rtol=atol=1e-6.
Failed references are excluded explicitly, generated errors count as failures.
This is validation pass@1, not the hidden public/private competition score.

Attach both private notebook outputs (T5 007 / Copy 008 version2) and the fixed
1M dataset version3. Accelerator: T4x2. Internet for pinned non-PyTorch packages.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, time
EXPECTED_MANIFEST_SHA = '7480c7d1778bffce50db987019f2ccf87c06f8643c1ce59ad811e4fd3c77d0e1'
INPUT = Path('/kaggle/input')
matches = [p for p in INPUT.rglob('bundle_manifest.json') if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA]
assert len(matches) == 1, f'Attach fixed dataset version3; found {len(matches)} matches'
BUNDLE = matches[0].parent
CHECKPOINTS = {}
for p in INPUT.rglob('complete.json'):
    value = json.loads(p.read_text())
    arch = value.get('architecture')
    if arch in ('copy', 't5') and value.get('global_step') == 3907:
        assert arch not in CHECKPOINTS, f'Duplicate saved outputs for {arch}'
        CHECKPOINTS[arch] = p.parent/'final'
assert set(CHECKPOINTS) == {'copy', 't5'}, f'Attach both saved version2 model outputs: {CHECKPOINTS}'
for arch, path in CHECKPOINTS.items():
    assert (path/'model.safetensors').is_file(), path
    print(arch, path)
print('Fixed dataset:', BUNDLE)


In [ ]:
# Only our checked local evaluator scripts are unpacked; keep Kaggle PyTorch/CUDA.
import base64, zlib
SOURCES = json.loads(zlib.decompress(base64.b64decode('eJzFPYty20aSv4JV6oqgA8GkRMmyvEytndhb3nVsl+3sXp2kQkHAUEIEAgwAWla0+vfr7nmgBwNQlOPLeWsjAPPq6enp9wxvd8TnOF/HTVYW4epm59jbOZX/W09m03P8776g/8b03wn9l77PEnpO6VnWfOqt4rr+29Sjt4VsTS8z2fCA/vuEvux3u5rttUPI59m+V5f5Z+GPA68FRdWbsR4dgNSwuoHHWoi2xf6RBG7KPsnihM025d3Sf/f5EDErkCNMQoXD4rRYVOXSS8o8FwniuPay5aqsGu/Hcl00ojIVimRdVaJowsW6WVfC1Pt0WYk4fV+W+csvIlk3pW6Sxk2c5IDutq75FHiLTORpfVqokl/rsjAvy7i5VJ2s4DHPznUH76lEvdTr81VVJqJuu6lvatWwyZZCt1qJahElej6mMkJTi6b24tpDUPB/fzMgnhb0x3tpyO/HslhkF8enhQf/VpVIM8IY9Jyvl8WxVzeVN/dOdy5EIaq4ESmUpAKxjA0qsRCAv0Q49XktBLtcN1EtAONpfewt8jJuoNpeOJE1rsvqSlRQkhX4fSa/LsWyrG6iPFtmTbQ816V7B4eqPP5C0ETnN40wjQ8PDvYPNXg5TPKziJoyB+gBznbsqdhVteJzIPd1M1TrSNaCCaxgDmlWyUn+h9YN/rwtCwE18Q8iG+umYuFF0aqsmygrAPbIr0W+GHu7P1A1hW38tygrr4hxVQvPP93pYOp0B3bgjjsL+d2F+3RnzPrGf7jOCN2FaOKmqQiOgEYc2xWzhZfVWVE32I9PzQLvHDbA2EMQy6an2CcsBYj3samGdB5m9QInrmpSmYTkr94EX3ya8xwJxZmyFxephnvuTbozonWNs1p4/8I6L6uqrPyFYp0p406MQ0g+aHHDmPE0zmQOvJZZqgLZ+AlrLL8fqQa3OJk74D7jwXVV9C3XrUPW6qNFy3/yQuIC8jWabolzOfMhNn/IMSYrTVuxpWRNYkktRyIctphWizUgepQslN3NDp2qVvET1muPFNq4lA6P1JvU5oTuCsLCdLDfs5YB8hazUj0V/vSV4Vtq0aoQEm9qTYTSQ7oFasdxVUZVtVEMqMEphg5ukQNQiSNobDQ4KJD8YM8F8pzN9lDOmVGgxSJmjD64HvbE0VIkJS08rt2kbGimvykwFGfZvEApg09BxiE4Z2MzmPTyscVYSHS3wqkpI0R0K5bwjeGUGEEN7OYWmPgyRGI67lCjKRjTDsFX3CFSBZId33U7PDndacXo6c4ZDFCrDsO2YGzoof0G24Z2BAlbkcNiS3lrCECA/laoYe7Xej6Iep03jtYDOkRahz9JFUrrIE2VJVgAVVxNoF8HOC1IAxCkOwof+XqA2iZqWo7u1cU/TB4b4Iw7ygJ9RpmpFSuXpcihkIccO8i5Pd2BWs0apMsxdLIu6vUKdUaj0yEfE7h9ZAWSqJae/Q34ydbsAYGh5URgEBF3Zs65KGiiITAE+AObfd0sQE/bAS3kB4XnsKMcfhtsuExij83zyEKDDcCGGVXiNyBa1DZvtd587MmFhPfVulWNjrFTfxpIJSsRWe6r2Xa0qPGYKRpKozg2mLH1D++RN53szR492lPwNNUNQ5eyRnCrGtMkrNaFb3PgE7BSQknw8XlOoO++lnPefYF/caPjbvGjaJHlIorG4XXWXEbIQmAFJaqgXzAzwCTGtTwLOgK0gG03R6MqTNfLVe0rxMFck3iF9lskt+b8U4VaTSO+qEe7H4WreT/qoLPrdN6BdBWjlcj6UcJLfEnEihtt4SfZ28svqwzYyr1kpwbvUNvC8gRIskq4JuTd9gN/fHGndtnmHWTWkYBC1DNIv/M+vv77f//4/pfH8Pefr9+8sY16KNjtyh8bOimLnvFmXBBv9mYoBXzS7q59rr0fGRm7N5nOPCAYREFEqAvbOUgce3OO4v6Jo9C63X0aeLt7szspWkCb491qbju0ivKpdwE5vxCtziF5hELKlFscfNoHbWPFWo4Y5qZM59B9uE4RpdF6t+7E7zaQiM0CKpKWgEzae2DzgYDX/dVNCsi1twNV+8fHd29/EjgQaWS8O1LWPkgUa3VNSUXQJPoxoGhEua9c1fSB1p5xnSGYqB155GBplQo55xD0ABDlMTz6BlMBVRvrekQQsjqX/b+t49yPkwb+BICYlUhAtNyjBaCZ1moBTKzrfowdx8ravqnUZTnNzUr3MEatgt51qzGZ27IUtW39fRMUPtiLgXSRjGX7XnCsagys/k7JBh3sC0u7Or+c3RDoLhmzRspJkeRlLXoWqRI5OlW0hHAdMAE6jXgV1xfDzBu1Ld59FhVg47q7H7aYTD/K8qzehDMqdskBtSdNDTAQvnaIIc99Tb9AVJpmpZ6PX5Bl/p6tHMSNx5ugRRV3A7RU7EIrW4dX4qb2xxwx+lMXYqp/AoVnLWDqlc8DvuA0FCKsvfyV+0VvfeXaF6TR+My4GN78tgVBrGDAVmFhAkeeWMao1rmVMaPFhO3G9bpSRRuvD44bmI6p31DbmsyEmDBLYWaxYMa/uVE9Y8DNrLE4pPeY2UyaqPFcT5PsYsEkzp4jYff3HhwKUZKXvewz3LruA9fB2NPRVgqF/KK0IRUbmg/YbkaDYAV7DFFMIeFKyOzgDwSY9DA6zMQNQ8cBY0WL+uxGSwPiPr2BleYaq3YLttS6ld9G+wklfYsucvYZMmfMk2xpuNsMpH0/C7YsVqCNTzdmiOdDH/D+AhayIW01wk3r+mAdf5eCnYOr94rBnUUaPaQmQu88TpWlp4dVC8tdsn2DtDUXbpxzSuXS2zelFtOe/fKVXj251s+866osLqK4qK+FxJqeOBkiDkz3ODrkl+RbEO/X0i3jGYb7t1KBcTo1lX3ny15LE2rfq+XyfNYHY+Qz5s+/N6rdkG1vcTEFpbMUrovU0vVJrAYOWVpLtM/Qzr2zrvdYARvzjThz5rXfxbuFsyNOChsi2lyOLpwKTALNuHw9Yii4Zxs5trVarqTbzJVS1iY6Uov2/qa5LItjr+1qxjERO1RwFHjON87KzgMXPefsvweb+1W8mEko1QcTb2oiFo+WeD9fZ3kDyqomPLIb6GWPqST7T114Z2xcrYyRGVOV9UpmJXA9pkewM44IIkOG92vX8N2anmfnx4xVc7ZwZHEvNpd9BojElJXZsbUyZKl6ri+IQwy8sOuKVQjnDE7tuq0zZazgrqVpONyiL8TL1IPp9tzWjkvaKTUTayG5z4OtpEsUCqAnrD6ngr7oNOsubrvjnvlOek1yKZIrUTlqoev2tnDLcWHJz9CyXOhZ2kGgk3YNIX8swx9UrAIg0icnvyn7stAqjIrOQE8UzmLGVqiKqMPLuKbIlWWMoX+cqqD3h8a4Vd4vlUMDvfIO2fOGPtmr6Zd/G4jbqCEDFn8yUWC00FX52HGjOTHPKTdkXG2aMSwdUH94nJcblR3z8bVeG2m2tnFes649UV5CRqHxHsqvFBmoN024J+tDsaYeqJ3AL+zB2yGI/lKxfA4WhZxTwE8HXVjIMhRfVnGRrmtRcRreMoIJ9QcqorOhLYJRsroxXo/ixmdlWSMq+OuP7yESNhKjiz+UuOGSimZ7AUu4sJZDddjDBofolyvoC8Zgpu13G7QhB4lkg/vtAsNWjoxZhMzE+P3spIMhGpW9wE5GkYVcg6Xm+Vbc3fiGqvLah//zpTKuGejBhI6hzskQkbYeLe41UyAPdNKdEvOKSZ7GESGZoh2wAsqM4s9xlmOUryeAYIVBWESB/KB5dpFBM4DNjHLSdn8mE8LKKx5wQcWfUGIaI+kbXPU3pzrKIcirKkjPAmv89quDThNG6h3QTmBRVS0YNNAdby+LTsn5qeCUnKuKQXEj5wEhKFpzJJK2tBd4a6FMjfBCYKRDFYw7kVMpx2WsI6LsVmjLArL9uBagz1Uiiusky+avYpgdEdt9a2h45AAQmpDbefYSlZ2cZWZsPm03Y91+eO79FDUwd0PMmyepieNYk0Z/U1xJYiy0JY9NlTvNejDQ3pPT7GOKgkpN1IEM9TrG/GGh6lle+aSsUiRdjC74uka4jFc+Z26BZpQ6JiDXg7irSsL2ZV/W6qNnXn5GXqsGM955e8X7+3LpYnOvGuiUEhyWbUcMo5t7MBzKaq6X7gzFOSYhbOxDu8UcIJzdfOb9xeyQTT22mvHto0eOwowkU6yXICHi5SqX2SFczTTlGjtEVvpFl3bJ06VhWU/PDmvq50CR99+mjLwfs+XIFuxlwzY53bF8inwM6K87KdeJZ7gnpafZTPfsKyHq87HiQD4Hx9ttuxt3QB0SxkMDtsdJYOKfga1eCGvFejDBmrD8Ilt1ARCVVtM3qESbrE2NZUqhQuY46OHS3crunu5hwL05t8eD1kR/Nu7xkFLXN15SLldxlSkmb00DrXgprfvWR+ZvRZcizpvLqCzym9Mdk+VFkqO1LbUmWYklLJmCpvZPKNeYJx2rjX0yOSPDUX3tU0TP1Obvb+1kdDBg7A9hnKYKIMrUD7wTxZLwbYAdnrWGXr8xw8ZnZsvyCo0WmWBVq8wtMnOi8opex0Ngh01JgtjH1DI25GOLukOskqNkR7ATWxbbWd6Y7SWnKxPhMeHbN26CUAr9QLNW6NDv7DwiMF1NZ2ipVNvxuJvAYMOsBx+H11WGiQTiS+Mz/UKl6rsKBXoVU0DefG+MxUCAWXExb3Mirdh2N6zs2yg1kwvYEkEPMJ1uhl7n0NrX+QK3i2dyO5UHLb9BWNU6U7ZFmLgn0v6NHek2XMprSIEAwjOPa/CAzMSBjrsBFbYTNgL3zatF8qxTic96bHOe7McwNJs405uw7rVn5v3NJ9iBl5bf2+reMfG5Y8Hy8HJnTydyUsi1fHC8nMMjuvjdZyEwDciwX/oP0KXZHjKQopekrAIXX1Zkhcd8eOYgc4/0+MVlnYVCGne9DMSwrTjeZHBnbuO5dkNqDAOzHrck91S5CYIDqJwdDEHkro+EIsHdZo6SqnOVcd2YZx0Vao+RIrNmJzpRJQGp2EQyOzKwPoBKuOmEqHopO+c/8X/P37x59++XP0Vv3/308iOliyvP+s9lutbemFfrgrShn8RCHROsLtZLQVqXepUPH0giyOfndZ1d6OeisF7XF+ZVj/fyy6pSo5Xq4d+XmYbgtRr4BdibV/LxR0BRVqxVhfd4JoOe3oLAk09vyjiVTx+B2IVphv7xhg39IiverWTpL0Vc3eiXF2DX6ucfSX0z0AC06nuc5/LpStyAoasGfN6AyDtfN6rBx/V5nVTZSiVlf8wzOnCpAXgDqoks+bRe6Sn/hIdDZHWhHrAeAmI+ty9Yu337u/SDlJWBE9XPSlyCqM9KjvZ/lFkh0o9Ni3sgGVDO/6XzUmG2RDL8+RWQqFr5PItr1t3zNDUzlg8/Yw6rAvGzGiQvy8q8AaGpJSyv1SKYXn4x3bwtG01KquxdxannN1NNP77RSGteKqQ0+q/68LowjfTj61r/lQOeFqBqf3z+6mX04pfXbz69fkt7xD67pLeuOk3XOdynAIzP9V7R9BIXN/IBU2zlU2qWPM0+LzVeBNi7Qpp7+ApKHR7Pls+YCCsfs0I/mAgQww/ogbI0N6S2jFfmmKh6yFSllV6JKi4uhD6MCDYgmeb0BjaXeqw1ddYUXlXPmqDqtaKUpqXs37MVgYaHub5j+U5KID5lvP3pVgkWSmRzpYlrMCyBypK/PF9mMIJvhSx4vOuQi3grI8BLATmiAgrJokUFlPB4EZ3HyRX8ucjLc1Cw9by5pBgKqFoHFnSsXPXzUIUE9IsF7fDH8k8ERMAzUiYtCNb8FKK53ulG7RWGD9gXnqZjndjiERSu9D1hABxpLD1gSXUWIEPckF57Xw5g4GQwWBF6drvEbH8w/4FpRBb2hlRrMRTDd9Eqs8j0Cuy5OQR8hawMMB7jUpMNFZd7/unTh9cvfvn00vA5i52hWY0qxViHm7EgJH9OjQ5hPHQF5uGd959WkYhXK6F5Bag05hm4lNBiRforNPPRIisXcaWl1+pGP60No4NN9sViTi0fYqwPc7pZUEk949lS9XhhGJg5mCHf16u05bog8tqeAEbsgI0Sa5GVZnUSazVgXUhpK/kzgkh6lK65UJ4eNfzNUlrKUbckq2tQIETTvq2wL/MOY/5aZpZOAww4U3jM2XPFnutVnmn8s2d6zEErUHPFnjWWV3msYcrLay2F1rDCeqHA4l+UecogSWLAFegJv6uGDfSuZ3wdr7CFlhmajLTcS9nbItOEU7FHRgOgnkHXBrsVf9ewSDoDRWiRGcrBL/V6Yb4gOi8yg1sSvlliykSSLeNcv8b56jJuXwot7WCNVrElgeW5UrXyon3+HeS56i7/da2Fc9U+JqIw8h60Xz30Mr7QO6L4dX1hCDWuIyS1C+mLzkqlY2RNBDrAhcYmvuutRDqOOWRUxwsRSStB+emUrJnLuFBeJu0L7grUKOaYhJTDJszn5nIN7Vek2ALZITuYfUK1nGwCqVda5+ydw+mcacrU6IMpE87bJVXe566gbLyuh0taUQZFwEcy5Ax+UwnQA8GMC59//NS9ioX4pjoMiFWu4/yKWnTOLdFREKw4DiNyuUaRTl6xjLP7rifoZK64+XffIr/ztgfazjUl9kmdguIyPiLgLVETPoG9qA55UXmYpb2tdCOV8kRVsakrcB5ydQPPQ7TSUnoyNl0PFRfZk43uhPsyJ+/HGhGWNiQlviQO4JsmkY7U3pZIttKoQAHQw/2luuvxFT1gi243W+Zo4PTRr2V4aoOFyE7J2o26x9O+PTFsl39suSO569o5UuqeYdG6/59AXJLrtplfZM2HfVxbFcU1FeKJ2h0YoWdN6Dwf1kXiNItXb7so35zpMz/YQ3CCHg5OgUvyhnWxQkUk0PDNxmA/vf5hPP6/IZLckjh1jdIwDD19B4wSjOh6jEjL1qGlVKyaS0snoC/eD97e5P68U75HldXCk/mtU2xcjO+zLcwzublP4eHHzKy804H7sNzDpjJvZ7jZdieaVWVZTUmC/nvRtqMPK7bG/d6DFyw+JIGTLw9PpHXPrXXuxdoSZ/130dyLaZQOgUeOqHFP+xNOv2jeKfL1vvem9oVEVOVsw1DtuWQ8RszKwYRU13GxA8POwul0PCg/9u6Fivq0QAvJuvX1ZUk9NMBPSW632BtvT7CdO4NXoQVbnGLTzhQ7BRyvVNhlHfKADferuOehEgZlzz2fWxz7mYlnMnG6S88W4oby2484nRtOiZfd0BUpkhic25pwd9M5FbLX/ftvXhLLVXNz7y1Dm69Zs4OeoUkzsa88QBPGm5OsBBO7lvdDsT2MkYYsp+w19UhWDwL1V5Of+YP2B4nEMAB1pYH/8aZo4i9EqQGj2sD7IJJ1hfELelfphMm9qKmpP52maSEHszygi/HJ8cFkctY7Xdvas0FtgdsWlnuuhNLwKEhI5KMfQd7jFEU61BBFWPv20SMrNIFdRcp2lzW4NX8ni6XNJqExy6EDAajYHJNYURB8560qvAL1fnLvC6UzalsEVizc5SlTLkB41gSn05l7SIw7v/nBKxU6HgJHe00fp+Lz42Kd50GfI5+JOTlV13qe6J4+PP857CzbSXfNzjCLsyJPHV5Up4tC+a2H+ijLtlyJwi9rsHM+K0h3rlFHBJoDyXIVdGPDPn4dOxFi+bmjH+AO9PWWDVrQOzeAkrwAiE25SnFWuWvufaHEwED4YbarT7V6r5ns3STLrAZQL3RiXB876zmes51hZpJfdu5scPRlqKqCtc9fxLV4SY94xHLLrd6578maBd6mqa4ASZgjBbY0fLg73dnAjzScjrJt2zaYmnlPppfMi4M1Kq8BgEKlr+F1d9PD/aPZw83o+1R0Lrq5ln3kSKKvuCzE/2F66P0zezEOHYHCZcgnwPofEydZQRJB3Qu3jUDp76e8YlEJ/KTWyD3rorWGZZwVftfN2N70Z27S8vHePNjzWTHWXJxtmon3/t3H1//90KyiAf8grO2P738JWJxOuVW5IGDOFOvUbE/k0tUNubLpXBFgdErH3rXDbYxFuOer7ajhkbdIw76dZxJx6nJdJdZFofIL3uxV0RWIvvn04c3rn19/ighH+nLBk84VjHR2qL9kPH7QMM8/WqPYlzVaw3SKHjjOq4+v/+clDDUJvMkDm7599+r1G2y7vxd4+3vjzl5lgYDAe/fR0QK5FJFkvcdNCCfgbKeMcu2CS/p91rjn6gQWVbUMFs71+pITr4G57iZ5mVzpCyKfed2G7jWH2vXn3t7HTCo7+stDvdK035uymwtXdGGtS9HmEj6yRxgNkmZ6xha2FlVGMbxUMxp9Qgl72EqwPESiDg33ABHbI171saEbfcWgTIv229GUiZbhVfsqFEOHx6IIeS+pzeooDLFimcRszijV8WeRWmnM7wqxWy4W+qoRgNeD6TaXwlsXySUm16Te3uTKUzYG4gJ2zjPQnmClYAjQgswvUOikweqCzK7BREFUEdXX9gcmBP+ZB/0eEFX+TkeGVdFlXOPPSHSSCe//hYmbNrUQ+2xlFhLW+bpIczzCVQKl0FUAq5K8XlKEBt4joJYquYS1SPDKVfRwfM4SMR8l6zQ+nozYeYrzuEkuoxoWa344C+hC3EJcR015BSQ43zs4BB0WjweJeFnPZ3hhHTCheeeMi75gDVYV1hpoPqov4/noyexokjxJp0+eHJ0vFok4mKTnT4+eTKZPF3tJsjh6kkwOF0eHsGWnUPg0To+mUzFbpPvJkyfpRExHfUNU5XU996cT+hfAasM/4326BzNA/HgIDlENDPOeymNG3ityDFKCBfLomrLUTGtzp17/z3uwogZzq1U4k34spIqLGlOHRGV+q+T5uil/BoaRvyqrj+K3Pfj/m5+tXpZYWiO//iAushrzIGgP0H7xMLcjOo+JtFsaCNmosn3YHESYmljQNlFdiwINjAiK6nQVszZX8cUFFOAyVBmlquomqCFFEhuBJ89FSCo3bduUdd2me6dEYN/Ip89EmmMqQ4fqNx2ofwUW2EtqpkJ1ow9isUZDCJbBwxNYBCyDDswF2e3dyLpWYOvzL7jywARw78CGAjmb4aGPrLnBs5gtmmzKsynbert3i/V+1acL1Sf7Tli/JXR1Y/LjEfAU0wEdjRmNQzyHKs+3aOpGozYXTStChjvUVYc7w7OEFEaSo56MFEoAfOA5IzpjrJHnFA50UQv1U0IDvbjlAx3JU0Gjs5MR30GyO/7Fam7Q87Wt1KkeHFasyuRStpzqn99hIk2eSzsZwadR6zUnzjxwlstq3L6EEiU+paX6SxDB1ElA1m5bbTzWy4ZEqc8Vs3JTGKXZhSQ5JfhCiWpt/9OhtPKaTqRBV3zPmsahTBPzublNp5UxDy3CnDG1A2sB9IZhapAHo2AUeKPj0Xj8/ej0FGlO3TvfUi+u6boyGMTzXnxZ5rbEbKl6jty+feWHH9uvEXDh1Zytpkzyoc+js4DhXG5zRDA+8N5YHYm1OUPKpfginzAzyNoP88F9wvru0v5807Yw+oL8E3AloX1kvbcagnlyVAn7FYYoQb9DdKkTcm1v6xqkQAy4VevMT3ai5TMfvQKVTEk7Rtl5fA5iDYHHlU/J1wcMHIQsKIdUxVIo2NlazNCbj9quniEMqZeuK5IWSnMk6s3LugYBWmRAd/CN9wiKaZaA4hvhpXgR9qAP/6kDuiBu51pXDIvy2tfqYrhuQK/O6lJmERuibaWpknePR+wA4yiwqVo1Ih0DDcYIF6OhGwVqf88qBV63RuoUIvVnuqgQDd4sAJujVwUJUZZHq0oQPqAhV5koi42u4q/pSK2UiiEi2bdZvhol1D90gMNEaFwgt/NHzcEImRnfjFhA3ylraWT0mxFjKvSTJkr0Inc58UcrlDn4HmUpsIe/wvsPo3Hgj87L2iqAd1kgOgWCCviZXH2nsDmrYM9G/bocs/PUpNu7iI1yEMqRmhIG8+kZD6uqeVhXHvcjTlJ5FRFtGbCHh9Dz3NTnZ1jFc9rcdj8w26gt0ws6sEwMXbY2qZFlk5se2+x6wC4Sj12pKX3JjDQ7B0PGmytqliW6QqiJiaydkVbiwT6syoKm5vl5vDwHS0i1x3ohq6F7kpdE4UhEeaYVu1peGYYGlBh2ThIDl5aDAzNCaOaykLINpoedbpl9qTELpAp6YVl7w4upqHmwXLDFLisgFHLFIxdS0t5IIOgEhOpcTo7OKX+RF01g6Qm9g0pCP+wBXYN+orkTv2xt7p0gVs68R6bp2MG8nl57CxNxwu5VTMT5ME4jscYuxiCBHmikd36iD9gvnpNH1YImOJFqDE1+zGWY86NfBUyDbiqhuieyo2P55/u23Znd7DpLAcY5/dzLRnTJVAcsQODUaN0Ij65vKAkMM1gv/2SoW+977wSI4OyRT4DsbrNi/F8fUF3J32lmrEWwFuqrzZDadSW4kz8TWH5Vl+Yi+ptvhpubp6Azvbn92nP3zz0KTq9ShI8yfAHC97c1XUsxn/Z03lWPOlpRd2UkO2+viAJVbiXwNgr16wH9VGfQisq0hEb/boBqELA+k9XaB+5akvbvJM7hv+/A6K7qRoth6ZVQgkru+oCsk1i6IHbbNZK3uciGoduxgW5uHk+mx2duRVGk8iaykhL9dOXtejzWTyEhxYdOxt9P6UYP2S/x6+E+GUNUhK0NDbmUqPrPJb7dxviP/MXQfN5ydYm9wc0CC3eVrSK8rpY0T0mKRCEDY2B39/QvFemv6dz+6QJ3GI27P9KzqoxUrbujSzVggSL6tSpYtjm9DnQFGovSl6QzEylSri86l6x+vR/mnW3e2UGp/EkINJ0decHlT6eZyONVPSQBd5V87Igb1xZYVeVFhT/wo6wBIjWDqHlKR0qasonzXntT/1uBrQ5GlWpIZpIGUEfO5uq9Ow900Emxiz8Di/m3hJD/8vwWDY+mk7FdrHgSwdPDRSh5w1+Mbrl2eccuwPNusZu7x7emlzv6hSgF5HE4XdzVI8zuXNeXnF326CLtrPVct1kUdkOQuQtW2kfEHBkjGFsNeq7DwWVsL7Cli3BGG+7BYQrxCBiyVvGAm+oio5yTs79FL9N1VXIbShOjcD2YvFqScmhMk5S5Gm3U6oz9a2tqSp8KyB2+uOwKJ9yiDGF9q2yicPavvJjrc5wLjfX1erOg+zPY8z22X9yfGJrjD1QHPT9OrQra+3nmGqmt22HU8YUS5aiftOKSZNQKD1Qm58r91tXinTvl1OVd1J+5MWiDx6tPX2knf4+y426juftpo6/LcmWcjJwKKI42+WxGHX9U53435nLbtZFyMrJu0bPcZljSFRd0+V95fTLqFIzOuHuTswDWobxED0+xS5dbuYjiPFeqXjrvg0w2GZ097uPhmKpeX34b1xK/Q2vUXqH1f8gftL+0wx6Y49cQbs8lnb1bX54vVJeG9wWfRzr2rB0VFP1F81hHgsPn6jaW91Tip0LeNYKKWRSlZRJFPXe4+SMW1RkFo9Y7Bi8SZfBghQa48JNQ0KVu+jIYf7S7O/qezr6ok8L+KII+dkdjmXmCv17JZz/QhzSPcIHkYWwTBt7cjCT4LkpwaEpeDPL06U4OZ5ubA4fYBQ6xKzlEbxewqzf3AeS/S4yot7kGAFogv1N90B/spdaiDUPmjx59hu8+fkZa27n7X/BV6n8=')))
EVAL_SOURCE = Path('/kaggle/working/evaluation_source')
EVAL_SOURCE.mkdir(exist_ok=True)
for name, source in SOURCES.items():
    (EVAL_SOURCE/name).write_text(source)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(BUNDLE/'requirements-kaggle.txt')], check=True)
import torch, transformers
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2'
print('torch:', torch.__version__, 'transformers:', transformers.__version__)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(2)])


In [ ]:
OUTPUT = Path('/kaggle/working/pass1_val20k_beam4_256')
OUTPUT.mkdir(exist_ok=False)
processes, streams = {}, {}
for gpu, arch in enumerate(('copy', 't5')):
    environment = os.environ | {'CUDA_VISIBLE_DEVICES': str(gpu), 'OMP_NUM_THREADS': '2',
        'MKL_NUM_THREADS': '2', 'OPENBLAS_NUM_THREADS': '1', 'TOKENIZERS_PARALLELISM': 'false',
        'HF_HUB_OFFLINE': '1', 'HF_DATASETS_OFFLINE': '1', 'PYTHONUNBUFFERED': '1',
        'PYTHONPATH': str(EVAL_SOURCE) + os.pathsep + str(BUNDLE)}
    command = [sys.executable, '-u', str(EVAL_SOURCE/'evaluate_saved.py'),
        '--bundle-root', str(BUNDLE), '--checkpoint', str(CHECKPOINTS[arch]),
        '--output', str(OUTPUT/arch), '--architecture', arch, '--device', 'cuda:0',
        '--batch-size', '64', '--num-beams', '4', '--max-new-tokens', '256']
    streams[arch] = (OUTPUT/f'{arch}.log').open('w')
    processes[arch] = subprocess.Popen(command, env=environment, stdout=streams[arch], stderr=subprocess.STDOUT)
    print('Started', arch, 'GPU', gpu, 'PID', processes[arch].pid, flush=True)
last = {}
while any(p.poll() is None for p in processes.values()):
    for arch, process in processes.items():
        progress = OUTPUT/arch/'progress.json'
        if progress.is_file():
            value = json.loads(progress.read_text())
            if value != last.get(arch):
                print(arch, value, flush=True)
                last[arch] = value
        if process.poll() not in (None, 0):
            print(arch, 'FAILED:', (OUTPUT/f'{arch}.log').read_text()[-12000:], flush=True)
            # Do not stop the other independent evaluation.
    time.sleep(20)
for stream in streams.values():
    stream.close()
codes = {arch: process.returncode for arch, process in processes.items()}
(OUTPUT/'exitcodes.json').write_text(json.dumps(codes, indent=2))
for arch, code in codes.items():
    if code:
        print((OUTPUT/f'{arch}.log').read_text()[-16000:])
assert all(code == 0 for code in codes.values()), codes


In [ ]:
results = {arch: json.loads((OUTPUT/arch/'metrics.json').read_text()) for arch in ('copy','t5')}
assert results['copy']['validation_sha256'] == results['t5']['validation_sha256']
assert results['copy']['num_examples'] == results['t5']['num_examples'] == 20000
assert results['copy']['reference_status_counts'] == results['t5']['reference_status_counts']
(OUTPUT/'summary.json').write_text(json.dumps(results, ensure_ascii=False, indent=2))
import pandas as pd
from IPython.display import display
display(pd.DataFrame({arch: {key: value[key] for key in ('pass@1','num_passed','num_evaluated',
    'num_reference_errors','bad_code_rate','num_hit_token_limit','generation_seconds')} for arch,value in results.items()}).T)
print(json.dumps(results, ensure_ascii=False, indent=2))
